# FIREGUARD ID v2 — Minggu 4: [M -> O] Tuning Model Baseline (K-Means k=2..8 Sweep)
**Sesuai Ketentuan Minggu 4:** Evaluasi sweep `k=2..8`, stabilitas multi-seed, dan seleksi berbasis keseimbangan klaster + validasi spasial.

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import json
import numpy as np
import pandas as pd
from src.utils import load_yaml, resolve_path

proj_cfg = load_yaml("config/project.yaml", repo_root=REPO_ROOT)
print("Repo root:", REPO_ROOT)

Repo root: FIREGUARD-ID


In [2]:
km_exp = json.loads(resolve_path("reports/clustering/kmeans_experiments.json", REPO_ROOT).read_text(encoding="utf-8"))
rows = [{
    "k": r["k"],
    "inertia": r["inertia_wcss"],
    "silhouette": r["internal_metrics"]["silhouette_score"],
    "davies_bouldin": r["internal_metrics"]["davies_bouldin_index"],
    "calinski_harabasz": r["internal_metrics"]["calinski_harabasz_index"],
    "multi_seed_ari": r["multi_seed_stability"]["mean_ari"],
    "spatial_transfer_ari": r["spatial_generalization"]["mean_transfer_ari"],
    "min_cluster_share": r["internal_metrics"]["min_cluster_share"],
} for r in km_exp["sweep_results"]]
display(pd.DataFrame(rows))
print("Selection justification:", km_exp["selection"]["justification"])

,k,inertia,silhouette,davies_bouldin,calinski_harabasz,multi_seed_ari,spatial_transfer_ari,min_cluster_share
0,2,1595.6014,0.543706,0.807048,968.7602,1.000000,0.972086,0.315358
1,3,859.7184,0.544178,0.714354,1261.7150,1.000000,0.972507,0.091442
2,4,614.6480,0.524246,0.805993,1287.9740,0.982952,0.929674,0.057444
3,5,473.1392,0.516500,0.819181,1316.8184,0.995779,0.948137,0.037515
4,6,401.4729,0.534711,0.825204,1270.2806,0.999050,0.871176,0.038687
5,7,349.6596,0.550538,0.779690,1234.8868,0.986055,0.970872,0.036342
6,8,313.5294,0.551179,0.770737,1192.9841,0.986463,0.887381,0.003517


Selection justification: Selected k=3 over k in [2, 3, 4, 5, 6, 7, 8] based on joint multi-criteria evaluation rather than a single metric: (1) achieves the lowest (best) Davies-Bouldin Index across all candidates (DBI=0.7144) and strong Silhouette (0.5442) with a 46.1% inertia drop from k=2; (2) attains exact 1.0000 multi-seed stability (mean_ari=1.0000, min_ari=1.0000 across 5 seeds); (3) achieves the highest spatial-block generalization transferability (mean_transfer_ari=0.9725, mean_test_dbi=0.7082); (4) satisfies cluster size balance (min_cluster_share=0.0914 >= 0.05), whereas k>=5 fragments into minority sliver clusters (<4% share) and k=2 collapses intermediate and chronic fire regimes.
